# F1 Optional - PyTorch Deep Dive (Nice to Know)
## Autograd Internals and the HuggingFace Trainer

This is the optional companion to `pytorch_refresher.ipynb`. The core refresher
gave you tensors, data loading, and `nn.Module`. That is everything the
fine-tuning topics strictly need.

This notebook goes one level deeper, for learners who finish early or want the
internals:
1. Autograd - what `.backward()` actually does, built up from a manual training loop.
2. HuggingFace Trainer - the production training loop teams use instead of a hand-written one.

It is fully self-contained: run it top to bottom on its own. Nothing here is
required for the rest of the course.


In [ ]:
# Disable TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Environment setup - runs in SageMaker Studio JupyterLab kernel
# No remote training in this notebook

import subprocess, sys

# Pin numpy<2 to avoid compatibility issues with older torch ops
# transformers and datasets needed for Section 5 (HuggingFace Trainer)
# sagemaker pinned to 2.x (3.x has breaking changes)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy<2",
    "sagemaker>=2.200.0,<3.0.0",
    "transformers>=4.35.0,<4.40.0",
    "tokenizers>=0.15.0,<0.20.0",
    "datasets>=2.18.0,<3.0.0",
], check=True)

import sagemaker
from sagemaker import get_execution_role
import boto3

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print("Environment OK - running in Studio kernel, no remote training needed for F1")

In [ ]:
import torch as pt
import torch.nn as nn
import numpy as np
import random
import matplotlib
matplotlib.use("Agg")          # SageMaker Studio: use non-interactive backend
import matplotlib.pyplot as plt

# Reproducibility seed
SEED = 42
pt.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"PyTorch version: {pt.__version__}")
print(f"CUDA available:  {pt.cuda.is_available()}")
device = pt.device("cuda" if pt.cuda.is_available() else "cpu")
print(f"Device:          {device}")

## Section 1 - Autograd: How the Model Learns from Mistakes

Our model starts with random weights. **Autograd** computes, for every weight,
whether increasing it would reduce the loss -- and by exactly how much.
One call to `.backward()` gives us all gradients at once.


In [ ]:
# --- Beat 1: What life looks like without autograd ---
#
# Suppose we have a single weight w and we want to minimize loss = (w*x - y)^2
# We need the gradient dL/dw = 2*(w*x - y)*x
# For one weight this is fine. For a 6-layer neural net with millions of weights,
# computing every partial derivative by hand is impossible.

import math

# A "model" with just one weight
w = 0.5      # our single parameter
x = 3.0      # one complaint feature value
y_true = 1.0 # ground truth label

# Forward pass
y_pred = w * x

# Loss (mean squared error)
loss = (y_pred - y_true) ** 2

# Gradient by hand: dL/dw = 2*(w*x - y)*x
gradient = 2 * (w * x - y_true) * x

print(f"y_pred: {y_pred:.4f}")
print(f"loss:   {loss:.4f}")
print(f"gradient by hand: {gradient:.4f}")

# Now imagine 6 features, 3 hidden layers, 64 units each...
# That is 6*64 + 64*64 + 64*64 + 64*3 = 8,576 parameters.
# Computing each partial derivative by hand would take days.
print("\nFor 8,576 parameters this approach is completely impractical.")
print("We need PyTorch autograd to compute ALL gradients in one backward() call.")

<!-- DIAGRAM: autograd-and-training-loop -->

```mermaid
graph TD
    x["x (input)"] --> mul["mul"]
    w["w (requires_grad=True)"] --> mul
    mul --> y["y_pred"]
    target["y_true"] --> loss_fn["MSE loss"]
    y --> loss_fn
    loss_fn --> loss["loss (scalar)"]
    loss --> backward[".backward()"]
    backward --> grad_w["w.grad"]
    grad_w --> step["w -= lr * w.grad"]
    step --> zero["w.grad.zero_()"]
    zero --> mul

    style backward fill:#fda,stroke:#c63
    style grad_w fill:#afa,stroke:#393
    style step fill:#dfd,stroke:#3a3
```

*Figure: Autograd builds a graph as you run tensor ops, then `.backward()` walks the graph in reverse to fill `.grad`. The same four steps - forward, loss, backward, step - repeat every batch, whether the model has 1 parameter or 1 billion.*

The diagram shows the autograd graph plus the training loop that wraps it.
Forward pass records every operation; `.backward()` traverses the graph in
reverse and fills `.grad` for every tensor with `requires_grad=True`. Then we
step, zero, and repeat. This is the heartbeat of every PyTorch model.

In [ ]:
# --- Beat 3: PyTorch autograd computes all gradients for us ---

# Step 1: Mark tensors we want to differentiate through
w = pt.tensor(0.5, requires_grad=True)   # our weight
x = pt.tensor(3.0)                       # input (no grad needed for data)
y_true = pt.tensor(1.0)

# Step 2: Forward pass - PyTorch records every operation
y_pred = w * x                           # records: mul(w, x)
loss = (y_pred - y_true) ** 2            # records: sub, pow

print(f"y_pred:   {y_pred.item():.4f}")
print(f"loss:     {loss.item():.4f}")
print(f"loss grad_fn: {loss.grad_fn}")   # shows PyTorch built the graph

# Step 3: Backward pass - PyTorch walks the graph and fills .grad
loss.backward()

print(f"\ndL/dw (autograd): {w.grad.item():.4f}")
# Should match our manual result: 2*(0.5*3 - 1)*3 = 2*(1.5-1)*3 = 3.0
print(f"dL/dw (manual):   {2 * (w.item() * x.item() - y_true.item()) * x.item():.4f}")

# Step 4: A minimal training loop
print("\n--- Manual training loop (5 steps) ---")
w = pt.tensor(0.5, requires_grad=True)
lr = 0.05    # learning rate

for step in range(5):
    # Forward
    y_pred = w * x
    loss = (y_pred - y_true) ** 2

    # Backward
    loss.backward()          # accumulates grad into w.grad

    # Weight update (inside no_grad so this op is NOT recorded)
    with pt.no_grad():
        w -= lr * w.grad     # gradient descent step
        w.grad.zero_()       # CRITICAL: zero out grad or it accumulates!

    print(f"  step {step}: loss={loss.item():.4f}  w={w.item():.4f}")

print(f"\nFinal w={w.item():.4f}  (true ratio y/x = {y_true.item()/x.item():.4f})")

# Step 5: no_grad for inference (fast, no memory overhead)
print("\n--- Inference (no gradient tracking) ---")
with pt.no_grad():
    inference_pred = w * x
    print(f"Prediction: {inference_pred.item():.4f}")
    print(f"inference_pred.grad_fn: {inference_pred.grad_fn}")  # None - tracking is off

### Lab 2 - The Manual Training Loop (Tier 1, ~20 min)

**Situation**: You have a single complaint feature (average sentence length) and
you want to learn a linear relationship: `severity_score = w * sentence_length + b`.
This is the simplest possible model - but it uses the exact same training loop as
a 100-layer transformer.

**Task**: Implement the complete training loop: forward -> loss -> backward -> update.

**Action**: Complete the steps marked `# YOUR CODE`.

**Result**: After training, your learned `w` and `b` should produce predictions
close to the true scores. The verification cell plots training loss.

Steps:
1. Create a synthetic dataset: `x_train` (100 sentence lengths, float32) and
   `y_train` (true severity scores = 0.7 * x_train + 0.3 + noise)
2. Initialize parameters `w` and `b` as float32 tensors with `requires_grad=True`
3. Implement the forward function: `y_pred = w * x_train + b`
4. Implement the MSE loss: `loss = ((y_pred - y_train)**2).mean()`
5. Call `loss.backward()`
6. Update `w` and `b` with learning rate 0.01 (inside `pt.no_grad()`)
7. Zero the gradients
8. Run 100 epochs and record `losses` (a Python list of float values)

**Stretch**: Plot the learned line on top of the scatter plot of (x_train, y_train).
Add a title and axis labels that reference the Barclays complaint context.

In [ ]:
# Lab 2 - Manual Training Loop (SOLUTION)
pt.manual_seed(SEED)

# Step 1: pt.rand gives uniform [0, 1] - good for normalized sentence lengths
x_train = pt.rand(100, dtype=pt.float32)
# y_train: true severity = 0.7 * length + 0.3 + small noise
# 0.05 * pt.randn(100) adds realistic measurement noise
y_train = 0.7 * x_train + 0.3 + 0.05 * pt.randn(100)

# Step 2: initialize both parameters at 0; requires_grad=True tells PyTorch
# to track operations on these tensors so .backward() can compute gradients
w = pt.tensor(0.0, requires_grad=True)
b = pt.tensor(0.0, requires_grad=True)

lr = 0.01
losses = []

for epoch in range(100):
    # Step 3: Forward pass - PyTorch records this multiplication and addition
    y_pred = w * x_train + b

    # Step 4: MSE loss - .mean() reduces the [100] tensor to a scalar
    loss = ((y_pred - y_train)**2).mean()

    # Step 5: Backward pass - computes dL/dw and dL/db and stores in .grad
    loss.backward()

    # Step 6: Update inside no_grad so these ops are NOT recorded in the graph
    with pt.no_grad():
        w -= lr * w.grad   # gradient descent: move w opposite to gradient
        b -= lr * b.grad

    # Step 7: Zero gradients - CRITICAL: without this, gradients accumulate
    # across steps and the update becomes incorrect
    w.grad.zero_()
    b.grad.zero_()

    losses.append(loss.item())

print(f"Final w={w.item():.4f} (true: 0.7000)")
print(f"Final b={b.item():.4f} (true: 0.3000)")
print(f"Final loss: {losses[-1]:.6f}")

In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
pt.manual_seed(SEED)
if x_train is None or not isinstance(w, pt.Tensor):
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    x_train = pt.rand(100, dtype=pt.float32)
    y_train = 0.7 * x_train + 0.3 + 0.05 * pt.randn(100)
    w = pt.tensor(0.0, requires_grad=True)
    b = pt.tensor(0.0, requires_grad=True)
    lr = 0.01
    losses = []
    for epoch in range(100):
        y_pred = w * x_train + b
        loss = ((y_pred - y_train)**2).mean()
        loss.backward()
        with pt.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
        losses.append(loss.item())

In [ ]:
# Verification - Lab 2
assert losses is not None and len(losses) == 100, "losses must have 100 values"
assert losses[-1] < losses[0], "Loss should decrease over training"
assert abs(w.item() - 0.7) < 0.15, f"w={w.item():.4f} too far from 0.7"
assert abs(b.item() - 0.3) < 0.15, f"b={b.item():.4f} too far from 0.3"

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Lab 2 - Training Loss (Barclays Complaint Severity Model)")
plt.tight_layout()
plt.savefig("/tmp/lab2_loss.png", dpi=80)
plt.show()
print(f"Lab 2 passed. Learned w={w.item():.4f}, b={b.item():.4f}")

**Homework Extension 2 - Autograd**

1. Modify the training loop to use **momentum**: instead of `w -= lr * w.grad`,
   implement `velocity = 0.9 * velocity + w.grad` and `w -= lr * velocity`.
   Does it converge faster?
2. Implement **gradient clipping**: before the update step, if `w.grad.abs() > 1.0`,
   scale it down to 1.0. This is used in transformers to prevent exploding gradients.
3. (Challenge) Implement the exact same training loop using `torch.optim.SGD` with
   momentum. Compare the final loss to your manual implementation. They should match
   if your momentum implementation was correct.

### Discussion (3 min) - Gradient Descent in Production

At Barclays, a fraud detection model is retrained every night on the previous day's
transactions. Consider:

1. The learning rate is a hyperparameter set by the team. Too high and training
   diverges; too low and it barely improves overnight. How would you find the right
   learning rate without running expensive experiments?
2. The training loop zeros gradients every step. What would happen if you forgot
   to zero them? Would the model still converge? Would it converge to the same place?
3. In production, should the fraud model do inference inside `pt.no_grad()`?
   What is the cost of forgetting this?

Share your answers with the person next to you. No wrong answers - the point is
to think about production tradeoffs.

## Section 2 - HuggingFace Trainer: The Production Training Loop

In production, teams use the **HuggingFace Trainer** instead of a manual loop.
It adds checkpointing, lr scheduling, mixed precision, and distributed training for free.
The one catch: Trainer expects `datasets.Dataset` objects (HuggingFace format),
not PyTorch `Dataset`. We convert using `HFDataset.from_dict()`.


In [ ]:
# Bridge cell - rebuild a small classifier, dataset and loader
# The HuggingFace Trainer demo below references `model`, `dataset` and `loader`.
# In the core notebook those came from the nn.Module section. This optional
# notebook is standalone, so we recreate them here.
from torch.utils.data import TensorDataset, DataLoader

class ComplaintClassifier(nn.Module):
    """Two-layer feedforward classifier for Barclays complaint categories."""
    def __init__(self, input_dim=6, hidden_dim=16, num_classes=3):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.relu(self.layer1(x))
        return self.layer2(h)

pt.manual_seed(SEED)
model = ComplaintClassifier().to(device)

pt.manual_seed(SEED)
X_data = pt.randn(300, 6, dtype=pt.float32).to(device)
y_data = pt.randint(0, 3, (300,), dtype=pt.long).to(device)
dataset = TensorDataset(X_data, y_data)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
print("Bridge ready: model, dataset and loader recreated for the Trainer demo.")


In [ ]:
# --- Beat 1: Passing a PyTorch Dataset to Trainer (wrong type) ---
from transformers import Trainer, TrainingArguments

# Trainer wraps any nn.Module - but it is strict about dataset type
# What happens when we pass a PyTorch TensorDataset?

try:
    bad_args = TrainingArguments(
        output_dir="/tmp/bad_trainer_test",
        num_train_epochs=1,
        per_device_train_batch_size=32,
        no_cuda=True,   # force CPU for this demo
    )
    # dataset (PyTorch TensorDataset) - THIS IS THE WRONG TYPE for Trainer
    bad_trainer = Trainer(
        model=model,    # model from Section 4
        args=bad_args,
        train_dataset=dataset,   # <-- PyTorch TensorDataset, not HF Dataset
    )
    bad_trainer.train()
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

print("\nTrainer expects a HuggingFace datasets.Dataset, not a PyTorch Dataset.")
print("We need to convert our data using datasets.Dataset.from_dict()")

In [ ]:
# --- Beat 3: HuggingFace Trainer for Complaint Classification ---
import numpy as np
from datasets import Dataset as HFDataset

# 1. Rebuild our model fresh (Trainer will manage the state)
pt.manual_seed(SEED)
hf_model = nn.Sequential(
    nn.Linear(6, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
).to(device)

# 2. Create HuggingFace datasets from our numpy arrays
#    Trainer expects dicts with string keys; "labels" is the magic key for loss
pt.manual_seed(SEED)
X_np = pt.randn(300, 6).numpy().astype(np.float32)
y_np = np.random.randint(0, 3, size=300).astype(np.int64)

# 80/20 train/eval split
split = int(0.8 * len(X_np))
hf_train = HFDataset.from_dict({"features": X_np[:split].tolist(),
                                  "labels":   y_np[:split].tolist()})
hf_eval  = HFDataset.from_dict({"features": X_np[split:].tolist(),
                                  "labels":   y_np[split:].tolist()})
hf_train = hf_train.with_format("torch")
hf_eval  = hf_eval.with_format("torch")

print(f"Train samples: {len(hf_train)}  Eval samples: {len(hf_eval)}")
print(f"Train columns: {hf_train.column_names}")

# 3. compute_metrics: inline numpy - NO evaluate library (incompatible with datasets 4.x)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean().item()
    return {"accuracy": accuracy}

# 4. Subclass Trainer to override compute_loss
#    (needed because our model input is a tensor named "features", not "input_ids")
class ComplaintTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        features = inputs["features"].to(pt.float32)
        logits = model(features)
        loss = nn.CrossEntropyLoss()(logits, labels)
        return (loss, logits) if return_outputs else loss

# 5. TrainingArguments: the production configuration
training_args = TrainingArguments(
    output_dir="/tmp/complaint_classifier",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",        # NOT evaluation_strategy (removed in transformers 4.41+)
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=10,
    no_cuda=(device.type == "cpu"),  # use GPU if available
)

# 6. Trainer: wraps training loop, checkpointing, logging
trainer = ComplaintTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_eval,
    compute_metrics=compute_metrics,
)

print("\n--- Training with HuggingFace Trainer (5 epochs) ---")
train_result = trainer.train()
print(f"\nTraining complete.")
print(f"  Runtime: {train_result.metrics.get('train_runtime', 'N/A'):.1f}s")
print(f"  Loss:    {train_result.metrics.get('train_loss', 'N/A'):.4f}")

# 7. Evaluate
eval_result = trainer.evaluate()
print(f"\nEval accuracy: {eval_result.get('eval_accuracy', 'N/A'):.3f}")

### Lab 5 - HuggingFace Trainer with Custom Model (Tier 1, ~20 min)

**Situation**: The Barclays NLP team wants checkpointing and automatic evaluation
on a 4-class complaint classifier (account, fraud, payment, general).

**Task**: Adapt the Trainer setup from Beat 3 to a new model (8 features, 4 classes)
and a larger dataset (1000 complaints). Train for 8 epochs. Report best eval accuracy.

**Action**: Complete the steps marked `# YOUR CODE`.

**Result**: The verification cell confirms training ran for 8 epochs and eval accuracy
is tracked in `trainer5.state.best_metric`.

Steps:
1. Create `X_lab5` (1000 samples, 8 features, float32 numpy array) and
   `y_lab5` (1000 labels, values 0-3, int64 numpy array)
2. Build `hf_train5` and `hf_eval5` using `HFDataset.from_dict()` with 80/20 split;
   call `.with_format("torch")` on both
3. Define `compute_metrics_lab5(eval_pred)` -- inline numpy, no evaluate library
4. Define `FourClassModel` as `nn.Sequential` (8->32->ReLU->16->ReLU->4) on `device`
5. Define `ComplaintTrainer5` subclassing `Trainer`, override `compute_loss` so it
   reads `inputs["features"]` and runs `CrossEntropyLoss`
6. Create `TrainingArguments` with `num_train_epochs=8`, `eval_strategy="epoch"`,
   `save_strategy="no"`, `no_cuda=(device.type=="cpu")`
7. Create `trainer5`, call `.train()` then `.evaluate()`

**Stretch**: Add per-class accuracy to `compute_metrics_lab5` using numpy masking.

In [ ]:
# Lab 5 - HuggingFace Trainer (SOLUTION)
pt.manual_seed(SEED)
np.random.seed(SEED)

# Step 1: Trainer works with numpy arrays, not PyTorch tensors
# astype(np.float32) matches what nn.Linear expects
X_lab5 = np.random.randn(1000, 8).astype(np.float32)
# astype(np.int64) = long in numpy - Trainer converts this to torch.long for loss
y_lab5 = np.random.randint(0, 4, size=1000).astype(np.int64)

# Step 2: HFDataset.from_dict converts numpy arrays to a HuggingFace Dataset
# "labels" is the magic key - Trainer looks for this when computing loss
split5 = 800
hf_train5 = HFDataset.from_dict({
    "features": X_lab5[:split5].tolist(),   # .tolist() converts numpy to Python list
    "labels":   y_lab5[:split5].tolist(),
}).with_format("torch")   # .with_format("torch") makes __getitem__ return tensors

hf_eval5 = HFDataset.from_dict({
    "features": X_lab5[split5:].tolist(),
    "labels":   y_lab5[split5:].tolist(),
}).with_format("torch")

# Step 3: Inline numpy metrics - NO evaluate library
def compute_metrics_lab5(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean().item()
    return {"accuracy": accuracy}

# Step 4: nn.Sequential is a quick way to chain layers
# No subclassing needed for simple architectures
FourClassModel = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
).to(device)

# Step 5: Override compute_loss because our model reads "features", not "input_ids"
class ComplaintTrainer5(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # inputs is a dict; pop "labels" so it is not passed to the model
        labels = inputs.pop("labels")
        # Cast features to float32 (Trainer may deliver them as float64)
        logits = model(inputs["features"].to(pt.float32))
        loss = nn.CrossEntropyLoss()(logits, labels)
        return (loss, logits) if return_outputs else loss

# Step 6: eval_strategy="epoch" (NOT evaluation_strategy - removed in transformers 4.41+)
training_args5 = TrainingArguments(
    output_dir="/tmp/lab5_solution",
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # evaluates at end of each epoch
    save_strategy="no",             # skip saving checkpoints for this lab
    metric_for_best_model="accuracy",
    greater_is_better=True,
    load_best_model_at_end=True,
    logging_steps=50,
    no_cuda=(device.type == "cpu"),
)

# Step 7: Wire everything together
trainer5 = ComplaintTrainer5(
    model=FourClassModel,
    args=training_args5,
    train_dataset=hf_train5,
    eval_dataset=hf_eval5,
    compute_metrics=compute_metrics_lab5,
)

trainer5.train()
trainer5.evaluate()
print(f"Best eval accuracy: {trainer5.state.best_metric}")

In [ ]:
# Lab 5 safety-net: run this if you did not finish Lab 5.
# SKIP this cell if you DID finish Lab 5.
pt.manual_seed(SEED)
np.random.seed(SEED)

if trainer5 is None:
    print("Using Lab 5 safety-net so the rest of the notebook can run.")

    X_lab5 = np.random.randn(1000, 8).astype(np.float32)
    y_lab5 = np.random.randint(0, 4, size=1000).astype(np.int64)

    _four_model = nn.Sequential(
        nn.Linear(8, 32), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 4),
    ).to(device)

    split5 = 800
    _hf_train = HFDataset.from_dict({"features": X_lab5[:split5].tolist(),
                                      "labels":   y_lab5[:split5].tolist()})
    _hf_eval  = HFDataset.from_dict({"features": X_lab5[split5:].tolist(),
                                      "labels":   y_lab5[split5:].tolist()})
    _hf_train = _hf_train.with_format("torch")
    _hf_eval  = _hf_eval.with_format("torch")

    def _compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"accuracy": (preds == labels).mean().item()}

    class _FourTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            logits = model(inputs["features"].to(pt.float32))
            loss = nn.CrossEntropyLoss()(logits, labels)
            return (loss, logits) if return_outputs else loss

    _args = TrainingArguments(
        output_dir="/tmp/lab5_safetynet",
        num_train_epochs=8,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        no_cuda=(device.type == "cpu"),
    )
    trainer5 = _FourTrainer(
        model=_four_model,
        args=_args,
        train_dataset=_hf_train,
        eval_dataset=_hf_eval,
        compute_metrics=_compute_metrics,
    )
    trainer5.train()
    trainer5.evaluate()

In [ ]:
# Verification - Lab 5
assert trainer5 is not None, "trainer5 not created"
assert trainer5.state.epoch == 8.0, \
    f"Expected 8 epochs, got {trainer5.state.epoch}"
best = trainer5.state.best_metric
assert best is not None, "No best metric recorded - did you set compute_metrics?"
print(f"Lab 5 passed.")
print(f"  Epochs completed: {int(trainer5.state.epoch)}")
print(f"  Best eval accuracy: {best:.3f}")
print("  eval_strategy='epoch' used (NOT evaluation_strategy - that was removed in 4.41+)")

**Homework Extension 5 - HuggingFace Trainer**

1. Add a `transformers.EarlyStoppingCallback` to `trainer5`. Set `early_stopping_patience=3`.
   Rerun training with 20 epochs - does it stop before epoch 20?
2. Enable mixed precision by adding `fp16=True` to `TrainingArguments` (only if you
   have a GPU). Measure the speedup vs the fp32 run. Note: fp16 training is the
   primary reason Trainer is preferred over manual loops in production.
3. (Challenge) Replace the synthetic features with TF-IDF encodings of real text.
   Write 20 synthetic complaint strings, vectorize with `TfidfVectorizer`, and use
   the matrix as `X_lab5`. Compare Trainer accuracy to the random-feature baseline.

## Wrap-Up - What You Explored

This optional notebook took you one level deeper than the core refresher:

| Component | What it does | Barclays use |
|-----------|-------------|--------------|
| Autograd | Tracks gradients through computation | Model learns from prediction errors |
| HF Trainer | Production training loop with eval + checkpointing | What the real models use |

### Key rules to remember

- `.backward()` walks the recorded graph in reverse and fills every `.grad`.
- Wrap weight updates in `pt.no_grad()` so the update itself is not recorded.
- Always zero gradients each step - they accumulate otherwise.
- HuggingFace Trainer expects a `datasets.Dataset`, not a PyTorch `Dataset`.

You have now seen the internals. The core refresher was already enough for the
fine-tuning topics - this was the nice-to-know layer on top.
